In [ ]:
"""
Modify the baseline fuelscape for proposed &/ completed fuels treatments
Maxwell.Cook@colostate.edu
"""

# standard imports
import os, sys

# custom functions
sys.path.append(os.getcwd()) # add code folder to system path
from code.Python.archive.__functions import *  # imports all custom functions

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 'EPSG:26913' # NAD83 UTM Zone 13N

print("Ready to go !")

## Load the fuelscape raster (multi-band geotiff)

** Reference 01-Baseline_Fuelscape.ipynb **

The baseline fuelscape is a multi-band geotiff representing the landscape topography and pre-treatment fuel conditions. It includes all bands needed to run FlamMap models for fire behavior outputs. Notably, this baseline fuelscape also includes CFRI-specific adjustments to the lodgepole pine EVT to better reflect expected fire behavior in this system. This adjustment was done in the downloading of the fuelscape from the LFPS REST API.

In [ ]:
# load the baseline fuelscape raster
lcp = list_files(projdir, "fuelscape_baseline.tif", recursive=True)
lcp = rxr.open_rasterio(lcp[0])
lcp

In [ ]:
lcp.rio.crs

## Load and rasterize the fuel treatments

** Reference 00-Treatment_Data-COFT.ipynb **

In order to make fuelscape adjustments based on theoretical or completed treatments, we will rasterize treatment polygons with specific information about the associated adjustments to the surface fuel model and canopy characteristics. In this workflow, treatment polygons were pulled from the CO Forest Tracker from 2022 and 2023 in the NoCo Fireshed QWRA extent. The canopy and surface effects (based on treatment type) are already added onto this file, but we need to dynamically adjust the surface fuel model on a per-pixel basis.

To make adjustments to the baseline fuelscape, we can either 1) adjust fuelscape only within treatment polygons, or 2) adjust and treat the entire fuelscape (for later zonal statistics). Future efforts may benefit from using annual change in LANDFIRE layers to extract an independent observation of the change in fuels within treatment polygons.

In [ ]:
# load the canopy+surface effects
can_eff_flat = list_files(projdir, "CanEff_flattened.gpkg", recursive=True)[0]
surf_eff_flat = list_files(projdir, "SurfEff_flattened.gpkg", recursive=True)[0]
print(f"Canopy effects flattened: {can_eff_flat}")
print(f"Surface effects flattened: {surf_eff_flat}")
if os.path.exists(can_eff_flat) and os.path.exists(surf_eff_flat):
    can_eff_flat = gpd.read_file(can_eff_flat)
    surf_eff_flat = gpd.read_file(surf_eff_flat)
print(f"\n{can_eff_flat['CanEff'].unique()}")
print(surf_eff_flat['SurfEff'].unique())

In [ ]:
# load the proposed or completed fuel treatments
fp = os.path.join(projdir, "data/spatial/treatments/NCFC/NCFC_FT_treatments_from2022.gpkg")
trt = gpd.read_file(fp)
# ensure the appropriate CRS for the treatment data
trt = trt.to_crs(lcp.rio.crs)
# ensure valid geometries
trt["geometry"] = trt.geometry.buffer(0)
trt.columns

### Canopy Modifications

In this section, we apply canopy modifications to the fuelscape where treatments overlap. Canopy modifications are defined in the "canopy_effects.csv" file (see 00-Treatment_Data.ipynb" for reference). Canopy effects are joined to the treatment table based on the treatment type and are used here to modify the CBD, CBH, CC, and CH rasters in the fuelscape.

In [ ]:
# load the canopy effects table
canopy_effects = pd.read_csv(os.path.join(projdir, 'data/tabular/fuel_mods/canopy_effects.csv'))
# create a lookup dictionary for treatment types
effect_dict = canopy_effects.set_index('Treatment').T.to_dict()
effect_dict

In [ ]:
# filter to only canopy treatments
canopy_trts = trt[(trt['CanEff'] != 'None') & (~trt['CanEff'].isnull())].copy()

# map treatments to numeric value for rasterization
canopy_trts['treat_code'] = canopy_trts['CanEff'].astype('category').cat.codes
code_lookup = dict(enumerate(canopy_trts['CanEff'].astype('category').cat.categories))
print("\n",code_lookup)

# rasterize it
treat_arr = rasterize_it(canopy_trts, to_img=lcp, attr='treat_code', fill_val=-1)

# save this out to check it
aoi = 'NCFC'
out_fp = os.path.join(projdir, f'data/spatial/treatments/{aoi}/canopy_treatments.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
treat_arr.rio.to_raster(out_fp)
print(out_fp)

In [ ]:
# make the adjustments to fuelscape
# define the bands (CC, CH, CBD, CBH)
band_map = {
    'cc_AF': 5,
    'ch_AF': 6,
    'cbh_AF': 7,
    'cbd_AF': 8,
}
modified_lcp = lcp.copy() # work with a copy

# map the bands, make the adjustments
for key, band_idx in band_map.items():
    print(f"Processing {key}/{band_idx}")
    # isolate the current raster band
    band = modified_lcp.sel(band=band_idx)
    # map the treatment codes
    for code, treatment in code_lookup.items():
        print(f"\tWorking on {code}/{treatment}")
        # make sure it is a valid treatment
        if treatment not in effect_dict:
            print(f"\t[INFO] Skipping treatment '{treatment}' (no effect data)")
            continue
        # create the treatment mask
        mask = (treat_arr == code)
        factor = effect_dict[treatment][key]
        # multiply the band by the correct factor
        band = band.where(~mask, np.floor(band * factor)) # ensure downward rounding with np.floor

    # assign new band to the stack
    modified_lcp.loc[dict(band=band_idx)] = band

    # check the values
    before = lcp.sel(band=band_idx).values
    after = modified_lcp.sel(band=band_idx).values
    print(f"Δ {key}: {(before != after).sum()} pixels changed")

# ensure the output has the correct band types
modified_lcp # check it

### Surface Modifications

Here we adjust surface fuel codes based on treatment types. The "surface_effects.csv" provides a lookup table for the adjustments to the FBFM40 fuel model based on the given treatment type. We need to adjust FBFM40 values for overlapping treatment polygons, keeping all other bands the same.

In [ ]:
# load the surface fuel modification table
surface_effects = pd.read_csv(os.path.join(projdir, 'data/tabular/fuel_mods/surface_effects.csv'))

# Set FBFM40 as index to create the lookup table
surface_lut = surface_effects.set_index('FBFM40')
# Convert each column (treatment) to a dict: {original_fuel_model: new_fuel_model}
surface_dict = {col: surface_lut[col].to_dict() for col in surface_lut.columns}

surface_effects

In [ ]:
# Filter to surface treatments
surface_trts = trt[(trt['SurfEff'] != 'None') & (~trt['SurfEff'].isnull())].copy()
# set 'Mastication' as rearrange
surface_trts.loc[surface_trts['SurfEff'] == 'Masticate', 'SurfEff'] = 'Rearrange'

# Assign a numeric code for each surface treatment type
surface_trts['treat_code'] = surface_trts['SurfEff'].astype('category').cat.codes
surface_code_lookup = dict(enumerate(surface_trts['SurfEff'].astype('category').cat.categories))
print("\n", surface_code_lookup)

# Rasterize treatment polygons to match fuelscape
surface_arr = rasterize_it(surface_trts, to_img=lcp, attr='treat_code', fill_val=-1)

# save this out to check.
out_fp = os.path.join(projdir, f'data/spatial/treatments/{aoi}/surface_treatments.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
surface_arr.rio.to_raster(out_fp)
print(f"Saved to: {out_fp}")

In [ ]:
# Copy fuelscape to modify
modified_lcp_ = modified_lcp.copy()

# Extract original FBFM40 band (Band 4 = FBFM40)
band_idx = 4
fbfm = modified_lcp_.sel(band=band_idx).copy()

# Loop over treatment types and remap surface fuel codes
for code, treatment in surface_code_lookup.items():
    print(f"Processing surface treatment: {treatment}")
    if treatment not in surface_dict:
        print(f"\t[INFO] No surface mapping found for '{treatment}'")
        continue

    # Treatment mask
    mask = (surface_arr == code)

    # Get remap dict for this treatment
    remap = surface_dict[treatment]

    # Apply remapping to pixels where mask is True
    original_values = fbfm.where(mask).values
    new_values = np.vectorize(remap.get)(original_values)

    # Replace values in the band (where mask)
    fbfm = fbfm.where(~mask, new_values)

# Assign updated band
modified_lcp_.loc[dict(band=band_idx)] = fbfm

print("Surface effects stamped in !")

In [ ]:
# convert to integer
modified_lcp_ = modified_lcp_.astype("int16")
# save the treated fuelscape raster
out_fp = os.path.join(projdir, f'data/spatial/fuelscape/{aoi}/treated/fuelscape_treated.tif')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
modified_lcp_.rio.to_raster(out_fp)
print(out_fp)

## Create a distance to treatment surface

To assess the spatial decay of treatment effects, we can create a euclidean distance raster representing the distance (m) from treatments. Eventually, coming up with a density map as well to represent the clustering of treatments and how they collectively effect fire behavior or reduced risk (i.e., does the spatial pattern of treatments on the landscape matter?).

In [ ]:
# create a binary raster for treated / not treated
trt['treated'] = 1  # flag for treatment
# rasterize the treatment polygons
trt_bin = rasterize_it(
    trt,
    to_img=modified_lcp_,
    attr='treated',
    fill_val=0  # untreated = 0
)

# calculate the Euclidean distance raster
# create a binary mask
mask = (trt_bin.values == 0)

# calculate the Euclidean distance
trt_dist = distance_transform_edt(mask)

# convert to a raster with the appropriate attributes
dist_arr = xr.DataArray(
    trt_dist,
    coords=trt_bin.coords,
    dims=trt_bin.dims,
    name="distance_to_treatment"
)
# write the CRS
dist_arr.rio.write_crs(trt_bin.rio.crs, inplace=True)

# make sure the data align
dist_arr = dist_arr.rio.reproject_match(modified_lcp)

# save the raster out.
out_fp = os.path.join(projdir, f"data/spatial/treatments/{aoi}/distance_to_treatment.tif")
dist_arr.rio.to_raster(out_fp)
print(f"Saved distance raster (meters): {out_fp}")